# Single-cell frustration analysis (V2)
GRNs are the same matrices as the multicellular study (`run_ising_Wbank_V3.jl`): tree, negtri, ICFF, CFF, negtritwin, negtritri, each in an **asymmetric** (directed) and a **symmetric** version.
Asymmetric and symmetric GRNs are plotted in separate figures. Colours match the multicellular notebooks; symmetric GRNs use the pastel version of the same colour.

In [ ]:
using Plots
using StatsPlots   # violin!
using LinearAlgebra
using Random
using StatsBase
using Statistics
using JLD2

include("../functions/functions_frustration_indexSC_OG.jl")
include("../../../Model/OG/functions/functions_SCising_model_ALLsweeps_OG.jl")

In [ ]:
base_path = "/home/twwoodw/my_folder/compute/data/results1D"
#base_path = "/data/scratch/projects/punim0638/woodward/nonrcep_casestudy"

function make_filenameSC(base_path, varname, K, β, σ, n_sweeps)
    return joinpath(base_path,
        "$(varname)1D_$(K)_$(β)_$(σ)_ALL$(n_sweeps)sweeps.jld2")
end

In [ ]:
# GRNs: same matrices as the multicellular study (run_ising_Wbank_V3.jl)
K_tree = Float64.([
    0  1  1  0  0  0  0  0  0;
    0  0  0  1  0  0  0  0  0;
    0  0  0  0  1  1  0  0  0;
    0  0  0  0  0  0 -1  0  0;
    0  0  0  0  0  0  0 -1  0;
    0  0  0  0  0  0  0  0 -1;
    0  0  0  0  0  0  0  0  0;
    0  0  0  0  0  0  0  0  0;
    0  0  0  0  0  0  0  0  0
])

K_negtri = Float64.([
    0   1  0  0  0  0  0  0  0;
    0   0  1  1  0  0  0  0  0;
   -1   0  0  0  0  1  0  0  0;
    0   0  0  0  0  0 -1  0  0;
    0   0  0  0  0  0  0 -1  0;
    0   0  0  0  0  0  0  0 -1;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0
])

K_ICFF = Float64.([
    0   1  -1  0  0  0  0  0  0;
    0   0   1  1  0  0  0  0  0;
    0   0   0  0  0  1  0  0  0;
    0   0   0  0  0  0 -1  0  0;
    0   0   0  0  0  0  0 -1  0;
    0   0   0  0  0  0  0  0 -1;
    0   0   0  0  0  0  0  0  0;
    0   0   0  0  0  0  0  0  0;
    0   0   0  0  0  0  0  0  0
])

K_CFF = Float64.([
    0   1   1  0  0  0  0  0  0;
    0   0   1  1  0  0  0  0  0;
    0   0   0  0  0  1  0  0  0;
    0   0   0  0  0  0 -1  0  0;
    0   0   0  0  0  0  0 -1  0;
    0   0   0  0  0  0  0  0 -1;
    0   0   0  0  0  0  0  0  0;
    0   0   0  0  0  0  0  0  0;
    0   0   0  0  0  0  0  0  0
])

K_negtritri = Float64.([
    0   -1  0  0  0  0  0  0  0;
    0   0  -1  1  0  0  0  0  0;
   -1   0  0  0  0  1  0  0  0;
    0   0  0  0  0  0 -1  0  0;
    0   0  0  0  0  0  0 -1  0;
    0   0  0  0  0  0  0  0 -1;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0
])

K_negtritwin = Float64.([
    0   -1  0  0  0  0  0  0  0;
    0   0  1  1  0  0  0  0  0;
   -1   0  0  0  0  1  0  0  0;
    0   0  0  0  0  0 -1  0  0;
    0   0  0  0  0  0  0 -1  0;
    0   0  0  0  0  0  0  0 -1;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0
])

In [ ]:
# symmetric versions
K_treesym       = Matrix(Symmetric(copy(K_tree), :U))       + Matrix(Symmetric(copy(K_tree), :L))
K_negtrisym     = Matrix(Symmetric(copy(K_negtri), :U))     + Matrix(Symmetric(copy(K_negtri), :L))
K_ICFFsym       = Matrix(Symmetric(copy(K_ICFF), :U))       + Matrix(Symmetric(copy(K_ICFF), :L))
K_CFFsym        = Matrix(Symmetric(copy(K_CFF), :U))        + Matrix(Symmetric(copy(K_CFF), :L))
K_negtritrisym  = Matrix(Symmetric(copy(K_negtritri), :U))  + Matrix(Symmetric(copy(K_negtritri), :L))
K_negtritwinsym = Matrix(Symmetric(copy(K_negtritwin), :U)) + Matrix(Symmetric(copy(K_negtritwin), :L));

grns = ["tree", "negtri", "ICFF", "CFF", "negtritwin", "negtritri"]

K_asym = Dict("tree" => K_tree,    "negtri" => K_negtri,    "ICFF" => K_ICFF,
              "CFF"  => K_CFF,     "negtritwin" => K_negtritwin,    "negtritri" => K_negtritri)
K_sym  = Dict("tree" => K_treesym, "negtri" => K_negtrisym, "ICFF" => K_ICFFsym,
              "CFF"  => K_CFFsym,  "negtritwin" => K_negtritwinsym, "negtritri" => K_negtritrisym)

K_of(name, sym)       = sym ? K_sym[name] : K_asym[name]
grn_label(name, sym)  = sym ? "$(name)sym" : name     # also the K_name the result files are saved under
sym_label(sym)        = sym ? "symmetric" : "asymmetric"
sym_tag(sym)          = sym ? "sym" : "asym"          # for plot file names

In [ ]:
# same colours as the multicellular study; symmetric GRNs use the pastel version
grn_colors = Dict(
    "tree"       => :orange,
    "tri"        => :green,
    "negtri"     => :teal,
    "ICFF"       => :purple,
    "CFF"        => :violet,
    "negtritwin" => :blue,
    "negtritri"  => :cyan
)

# mix a colour with white; w = weight of the original colour
pastel(c; w = 0.45) = (rgb = parse(Plots.Colorant, string(c));
    Plots.RGB(w * Plots.red(rgb) + 1 - w, w * Plots.green(rgb) + 1 - w, w * Plots.blue(rgb) + 1 - w))

grn_colors_sym = Dict(name => pastel(c) for (name, c) in grn_colors)

color_of(name, sym)   = sym ? grn_colors_sym[name] : grn_colors[name]
color_grad(name, sym) = cgrad([:white, color_of(name, sym)])   # for heatmaps

# panels on a grid with 3 columns
function panel_grid(ps; panel_size = (400, 350), kw...)
    nc = min(3, length(ps)); nr = cld(length(ps), nc)
    plot(ps...; layout = (nr, nc), size = (panel_size[1] * nc, panel_size[2] * nr), kw...)
end

In [ ]:
### ----------- constants ----------- ###
n_cells = 400
n_genes = 9 
#save_every = 10 # save spin_lattice[1:save_every:n_sweeps] 


### spin lattice ###
ξ1 = [1,1,1,-1,-1,-1,-1,-1,-1]
ξ2 = [-1,-1,-1,1,1,1,-1,-1,-1]
ξ3 = [-1,-1,-1,-1,-1,-1,1,1,1]
ξ  = hcat(ξ1, ξ2, ξ3)

σ_neg1 = -1*ones(n_cells, n_genes) # all spin down
σ_pos1 = 1*ones(n_cells, n_genes)  # all spin down
σ_rand= rand([-1, 1], n_cells, n_genes)

half_ξ1 = repeat(reshape(ξ1, 1, n_genes), Int(n_cells / 2), 1)
half_ξ2 = repeat(reshape(ξ2, 1, n_genes), Int(n_cells / 2), 1)
half_ξ3 = repeat(reshape(ξ3, 1, n_genes), Int(n_cells / 2), 1)

σ_ξ1ξ2 = vcat(half_ξ1, half_ξ2);
σ_ξ2ξ1 = vcat(half_ξ2, half_ξ1);

In [ ]:
β = Inf
β_name = "βInf"

σ = σ_rand
σ_name = "σrand"
#σ = σ_ξ1ξ2
#σ_name = "σξ1ξ2"

n_sims   = 20
max_sweeps = 10_000
max_sweeps_name = "$(max_sweeps)"

# Load results
Every GRN, asymmetric and symmetric. GRNs without result files are skipped with a warning.

In [ ]:
SC = Dict{Tuple{String, Bool}, NamedTuple}()     # (GRN name, sym) => results

for sym in (false, true), name in grns
    fname = var -> make_filenameSC(base_path, var, grn_label(name, sym), β_name, σ_name, max_sweeps)
    vars  = ["sigma_histories_selfish", "sigma_finals_selfish", "E_histories_selfish"]

    missing_files = filter(!isfile, fname.(vars))
    if !isempty(missing_files)
        @warn "skipping $(grn_label(name, sym)): missing $(basename.(missing_files))"
        continue
    end

    SC[(name, sym)] = (
        σ_histories = load(fname("sigma_histories_selfish"), "val"),
        σ_finals    = load(fname("sigma_finals_selfish"),    "val"),
        E_histories = load(fname("E_histories_selfish"),     "val"),
    )
end

loaded_grns(sym) = [name for name in grns if haskey(SC, (name, sym))]
println("asymmetric: ", loaded_grns(false))
println("symmetric:  ", loaded_grns(true))

# Cell frustration in the tissue at a few sweeps (one GRN, one sim)

In [ ]:
##### VARIABLES #####
GRN_name = "tree"
sym      = false      # true → symmetric version
test_sim = 10
#####################

σ_histories = SC[(GRN_name, sym)].σ_histories
K_gene      = K_of(GRN_name, sym)
sim_name    = "sim$(test_sim)"

sweeps = [1, 5, length(σ_histories[test_sim])]
sweeps = unique(filter(s -> s ≤ length(σ_histories[test_sim]), sweeps))   # safety filter

plots = []
avg_frus = Float64[]

for sweep in sweeps
    frus_idx = eval_KGRN(K_gene, σ_histories[test_sim][sweep], n_cells)[2]
    frus_mtx = reshape(frus_idx, (20, 20))
    push!(avg_frus, mean(frus_mtx))

    push!(plots, heatmap(frus_mtx, title = "sweep = $(sweep)", aspect_ratio = 1,
                         colorbar = true, color = color_grad(GRN_name, sym)))
end

final_plot = plot(
    plots...,
    layout = (1, length(plots)),
    size = (1200, 500),
    plot_title = "SC intracellular frustration, 2D tissue - $(grn_label(GRN_name, sym)) ($(sym_label(sym))) | $(sim_name) | $(σ_name)"
)

#savefig(final_plot, "~/my_folder/compute/data/results1D/plots1D/plots_set1/mini_2D_tissue_SCintracellular_frustration_$(grn_label(GRN_name, sym))_$(sim_name)_$(σ_name).png")
display(final_plot)

# Final tissues

In [ ]:
include("../functions/functions_plotting_OG.jl")

color_key = create_configuration_color_key(n_genes);

In [ ]:
test_sim = 2
sim_name = "sim$(test_sim)"

for sym in (false, true)
    names = loaded_grns(sym)
    isempty(names) && continue

    ps = [plot!(single_configuration_plot(SC[(name, sym)].σ_histories;
                    sim = test_sim, sweep = length(SC[(name, sym)].σ_histories[test_sim])),
                title = grn_label(name, sym))
          for name in names]

    plt = panel_grid(ps; plot_title = "Final tissues, $(sym_label(sym)) GRNs | $(sim_name) | $(σ_name)")
    #savefig(plt, "~/my_folder/compute/data/results1D/plots1D/plots_set1/isolated_final_tissues_$(sym_tag(sym))_$(sim_name)_$(σ_name).png")
    display(plt)
end

# Isolated cell frustration density of the final tissues

In [ ]:
# frustration density of every cell in the final tissue of every sim
final_frus = Dict(key => reduce(vcat, [eval_KGRN(K_of(key...), σh[end], n_cells)[2] for σh in r.σ_histories])
                  for (key, r) in SC)

In [ ]:
for sym in (false, true)
    names = loaded_grns(sym)
    isempty(names) && continue

    v = plot(
        title  = "Isolated cell frustration density, $(sym_label(sym)) GRNs | $(σ_name)",
        xlabel = "GRN ($(sym_label(sym)))",
        ylabel = "Cell frustration density",
        legend = false
    )

    for (i, name) in enumerate(names)
        vals = final_frus[(name, sym)]
        if length(unique(vals)) == 1    # a violin of a single value is invisible: draw a point
            scatter!(v, [i], [first(vals)], color = color_of(name, sym), markersize = 6)
        else
            violin!(v, fill(i, length(vals)), vals, color = color_of(name, sym), linewidth = 0)
        end
    end
    xticks!(v, 1:length(names), grn_label.(names, sym))

    #savefig(v, "~/my_folder/compute/data/results1D/plots1D/plots_set1/frustration_density_violinplot_SC_$(sym_tag(sym))_$(σ_name)_$(β_name).png")
    display(v)
end

# Count the number of times a gene flips its activation state. 

In [ ]:
function average_flip_count(
    σ_histories;
    n_cells = n_cells,
    n_genes = n_genes,
    max_sweeps = nothing
)
    n_sims = length(σ_histories)

    if isnothing(max_sweeps)
        max_sweeps = maximum(length.(σ_histories)) - 1
    end

    max_actual_sweeps = maximum(length.(σ_histories)) - 1
    n_sweeps = min(max_sweeps, max_actual_sweeps)

    flip_counts = fill(NaN, n_sweeps, n_sims)

    site_flip_counts = [
        zeros(Int, n_cells, n_genes)
        for _ in 1:n_sims
    ]

    for sim in 1:n_sims
        n_sweeps_sim = min(
            length(σ_histories[sim]) - 1,
            n_sweeps
        )

        for sweep in 1:n_sweeps_sim
            σ_test1 = σ_histories[sim][sweep]
            σ_test2 = σ_histories[sim][sweep + 1]

            flip_counts[sweep, sim] =
                count(!iszero, σ_test1 - σ_test2)

            site_flip_counts[sim] .+= (σ_test1 .!= σ_test2)
        end
    end

    avg_flip_count = [
        begin
            vals = filter(!isnan, flip_counts[sweep, :])
            isempty(vals) ? NaN : mean(vals)
        end
        for sweep in 1:n_sweeps
    ]

    # Remove trailing sweeps where no simulation has data
    last_valid = findlast(!isnan, avg_flip_count)

    if !isnothing(last_valid)
        avg_flip_count = avg_flip_count[1:last_valid]
    else
        avg_flip_count = Float64[]
    end

    avg_site_flip_counts = sum(site_flip_counts) ./ n_sims

    return avg_flip_count, avg_site_flip_counts
end


In [ ]:
# (avg flips per sweep, avg flips per site) for every GRN
flips = Dict(key => average_flip_count(r.σ_histories) for (key, r) in SC)

# avg flips per gene as a 3×3 grid (genes 1–9 row by row)
avg_gene_flips = Dict(key => reshape(mean(f[2], dims = 1), (3, 3))' for (key, f) in flips)

In [ ]:
for sym in (false, true)
    names = loaded_grns(sym)
    isempty(names) && continue

    plt = plot(
        xlabel = "Sweeps",
        ylabel = "Average number of flips",
        title  = "Single cell gene flips per sweep",
        xlims  = (0, 30)
    )

    for name in names
        f = flips[(name, sym)][1] ./ n_cells 
        plot!(plt, 1:length(f), f, label = grn_label(name, sym), color = color_of(name, sym),
              alpha = 0.8, linewidth = 2, dpi = 600)
    end

    savefig(plt, "~/my_folder/compute/data/results1D/plots1D/plots_set1/SC_avg_flips_$(sym_tag(sym))_$(σ_name).png")
    display(plt)
end

# Gene flip rate 

In [ ]:
"""
gene_flip_rate(σ_histories)

Flip rate of every gene at every sweep: the fraction of the cells in which that gene flipped
between sweep s and s+1, averaged over the sims still running at that sweep.
Returns a (sweeps × genes) matrix.
"""
function gene_flip_rate(σ_histories)
    n_sweeps = maximum(length.(σ_histories)) - 1
    n_g      = size(σ_histories[1][1], 2)
    rate     = fill(NaN, n_sweeps, n_g, length(σ_histories))
    for (sim, σh) in enumerate(σ_histories), sweep in 1:length(σh) - 1
        rate[sweep, :, sim] .= vec(mean(σh[sweep] .!= σh[sweep + 1], dims = 1))   # mean over the cells
    end
    return [begin
                vals = filter(!isnan, rate[sweep, g, :])
                isempty(vals) ? NaN : mean(vals)
            end for sweep in 1:n_sweeps, g in 1:n_g]
end

gene_rates = Dict(key => gene_flip_rate(r.σ_histories) for (key, r) in SC)


In [ ]:
gene_xmax   = 30                      # sweeps shown
gene_colors = palette(:tab10)[1:9]    # one colour per gene, the same in every panel

for sym in (false, true)
    names = loaded_grns(sym)
    isempty(names) && continue

    ps = []
    for name in names
        R = gene_rates[(name, sym)]
        p = plot(title = grn_label(name, sym), xlabel = "Sweep", ylabel = "Gene flip rate (per cell)",
                 xlims = (0, gene_xmax), legend = name == first(names) ? :topright : false, dpi = 600)
        for g in 1:size(R, 2)
            valid_idx = findall(!isnan, R[:, g])
            plot!(p, valid_idx, R[valid_idx, g]; label = "gene $(g)", color = gene_colors[g], linewidth = 3)
        end
        push!(ps, p)
    end

    plt = panel_grid(ps; panel_size = (400, 400),
                     plot_title = "Gene flip rate over sweeps, $(sym_label(sym)) GRNs | $(σ_name)")
    #savefig(plt, "~/my_folder/compute/data/results1D/plots1D/SC_gene_flip_rate_$(sym_tag(sym))_$(σ_name).png")
    display(plt)
end


# Tissue energy

In [ ]:
function histories_to_matrix(histories)
    max_len = maximum(length, histories)

    M = fill(NaN, max_len, length(histories))

    for sim in eachindex(histories)
        M[1:length(histories[sim]), sim] .= histories[sim]
    end

    return M
end


In [ ]:
E_matrices = Dict(key => histories_to_matrix(r.E_histories) for (key, r) in SC)

# mean over the sims still running at each sweep
avg_E = Dict(key => [begin
                         vals = filter(!isnan, M[sweep, :])
                         isempty(vals) ? NaN : mean(vals)
                     end for sweep in 1:size(M, 1)]
             for (key, M) in E_matrices)

In [ ]:
for sym in (false, true)
    names = loaded_grns(sym)
    isempty(names) && continue

    ps = []
    for name in names
        M = E_matrices[(name, sym)]
        c = color_of(name, sym)
        p = plot(title = grn_label(name, sym), xlabel = "Sweep", ylabel = "Energy", legend = false)

        # individual simulations
        for sim in 1:size(M, 2)
            valid_idx = findall(!isnan, M[:, sim])
            isempty(valid_idx) && continue
            plot!(p, valid_idx, M[valid_idx, sim]; alpha = 0.3, color = c)
        end

        # mean trajectory on top
        avg = avg_E[(name, sym)]
        valid_idx = findall(!isnan, avg)
        plot!(p, valid_idx, avg[valid_idx]; color = c, linewidth = 2.5)

        push!(ps, p)
    end

    plt = plot(ps..., layout = (2, 6), size = (3500, 900), dpi = 600,
               plot_title = "Tissue energy trajectories, $(sym_label(sym)) GRNs | $(σ_name)",
               left_margin = 10Plots.mm, bottom_margin = 10Plots.mm,
               top_margin = 10Plots.mm, right_margin = 5Plots.mm,
               titlefontsize = 10)
    #savefig(plt, "~/my_folder/compute/data/results1D/plots1D/plots_set1/SC_tissue_energy_trajectories_$(sym_tag(sym))_$(σ_name).png")
    display(plt)
end

# Isolated Frurstration 

In [ ]:
"""
tissue_frustration_history(K, σ_history)

Tissue-mean intracellular frustration density at every sweep: for each cell, the fraction of its
GRN edges K[i,j] (i ≠ j) with K[i,j] σ_i σ_j < 0, averaged over the cells.
"""
function tissue_frustration_history(K, σ_history)
    pairs = [(i, j, sign(K[i, j])) for i in axes(K, 1), j in axes(K, 2) if i != j && !iszero(K[i, j])]
    nE = length(pairs)
    return [mean(count(((i, j, s),) -> s * σ[c, i] * σ[c, j] < 0, pairs) for c in axes(σ, 1)) / nE
            for σ in σ_history]
end

frus_matrices = Dict(key => histories_to_matrix([tissue_frustration_history(K_of(key...), σh) for σh in r.σ_histories])
                     for (key, r) in SC)

# mean over the sims still running at each sweep
avg_frus = Dict(key => [begin
                            vals = filter(!isnan, M[sweep, :])
                            isempty(vals) ? NaN : mean(vals)
                        end for sweep in 1:size(M, 1)]
                for (key, M) in frus_matrices)

In [ ]:
frus_xmax = nothing    # e.g. 50 to zoom in on the first sweeps

# one panel per GRN: every sim (faint) and the mean
for sym in (false, true)
    names = loaded_grns(sym)
    isempty(names) && continue

    ps = []
    for name in names
        M = frus_matrices[(name, sym)]
        c = color_of(name, sym)
        p = plot(title = grn_label(name, sym), xlabel = "Sweep", ylabel = "Intracellular frustration density",
                 legend = false, xlims = frus_xmax === nothing ? :auto : (0, frus_xmax))

        for sim in 1:size(M, 2)
            valid_idx = findall(!isnan, M[:, sim])
            isempty(valid_idx) && continue
            plot!(p, valid_idx, M[valid_idx, sim]; alpha = 0.3, color = c)
        end

        avg = avg_frus[(name, sym)]
        valid_idx = findall(!isnan, avg)
        plot!(p, valid_idx, avg[valid_idx]; color = c, linewidth = 2.5)

        push!(ps, p)
    end

    plt = panel_grid(ps; panel_size = (500, 400),
                     plot_title = "Intracellular frustration density over sweeps, $(sym_label(sym)) GRNs | $(σ_name)")
    #savefig(plt, "~/my_folder/compute/data/results1D/plots1D/SC_frustration_trajectories_$(sym_tag(sym))_$(σ_name).png")
    display(plt)
end

In [ ]:
for sym in (false, true)
    names = loaded_grns(sym)
    isempty(names) && continue

    plt = plot(
        #alpha  = 0.5,
        dpi = 600,
        xlabel = "Sweep",
        ylabel = "Intracellular frustration density", #(mean over sims)",
        title  = "$(sym_label(sym)) GRNs | $(σ_name)",
        xlims  = (0,30) #frus_xmax === nothing ? :auto : (0, frus_xmax)
    )

    for name in names
        avg = avg_frus[(name, sym)]
        valid_idx = findall(!isnan, avg)
        plot!(plt, valid_idx, avg[valid_idx]; label = grn_label(name, sym), color = color_of(name, sym), linewidth = 2.5)
    end

    
    
    savefig(plt, "~/my_folder/compute/data/results1D/plots1D/SC_avg_frustration_vs_sweep_$(sym_tag(sym))_$(σ_name).png")
    display(plt)
end

# Number of cell types per GRN

In [ ]:
# number of unique gene expression profiles (rows of σ) in a tissue
n_unique_profiles(σ) = length(unique(eachrow(σ)))

# one value per sim: unique profiles in that sim's final tissue
final_n_unique = Dict(key => [n_unique_profiles(σh[end]) for σh in r.σ_histories] for (key, r) in SC)

In [ ]:
for sym in (false, true)
    names = loaded_grns(sym)
    isempty(names) && continue

    plt = plot(
        title  = "Unique configurations in the final tissue, $(sym_label(sym)) GRNs | $(σ_name)",
        xlabel = "GRN ($(sym_label(sym)))",
        ylabel = "Unique gene expression profiles",
        legend = false
    )

    for (i, name) in enumerate(names)
        n = final_n_unique[(name, sym)]
        x = i .+ 0.15 .* (rand(length(n)) .- 0.5)     # small horizontal jitter so equal values don't overlap
        scatter!(plt, x, n, color = color_of(name, sym), alpha = 0.7, markersize = 5, markerstrokewidth = 0)
        scatter!(plt, [i + 0.25], [mean(n)], yerror = length(n) > 1 ? [std(n)] : [0.0],
                 color = :black, markersize = 5)       # mean ± std over sims
    end
    xticks!(plt, 1:length(names), grn_label.(names, sym))

    #savefig(plt, "~/my_folder/compute/data/results1D/plots1D/SC_n_unique_final_$(sym_tag(sym))_$(σ_name).png")
    display(plt)
end


In [ ]:
# number of unique gene expression profiles (rows of σ) in a tissue
n_unique_profiles(σ) = length(unique(eachrow(σ)))

"""
n_unique_history(σ_histories)

Unique profiles in the tissue at every sweep of every sim, as a (sweeps × sims) matrix.
A sim that stopped early keeps its final count for the remaining sweeps, so the mean at
every sweep is over all sims.
"""
function n_unique_history(σ_histories)
    n_sweeps = maximum(length.(σ_histories))
    M = Matrix{Float64}(undef, n_sweeps, length(σ_histories))
    for (sim, σh) in enumerate(σ_histories)
        counts = n_unique_profiles.(σh)
        M[1:length(counts), sim] .= counts
        M[length(counts)+1:end, sim] .= counts[end]    # finished sim: keeps its final tissue
    end
    return M
end

unique_matrices = Dict(key => n_unique_history(r.σ_histories) for (key, r) in SC)
avg_unique      = Dict(key => vec(mean(M, dims = 2)) for (key, M) in unique_matrices)


In [ ]:
unique_xmax = 30    # e.g. 50 to zoom in on the first sweeps
unique_ymax = 20         # cap the y-axis so the GRNs separate (the random start is ~400)

for sym in (false, true)
    names = loaded_grns(sym)
    isempty(names) && continue

    plt = plot(
        xlabel = "Sweep",
        ylabel = "Avg unique configurations",
        title  = "Unique configurations over sweeps",
        xlims  = unique_xmax === nothing ? :auto : (0, unique_xmax),
        ylims  = (0, unique_ymax),
        legend = :topright,
        dpi    = 600
    )

    for name in names
        M   = unique_matrices[(name, sym)]
        avg = avg_unique[(name, sym)]
        sd  = size(M, 2) > 1 ? vec(std(M, dims = 2)) : zeros(length(avg))
        plot!(plt, 1:length(avg), avg; ribbon = sd, fillalpha = 0.15,
              label = grn_label(name, sym), color = color_of(name, sym), linewidth = 2.5)
    end

    savefig(plt, "~/my_folder/compute/data/results1D/plots1D/SC_n_unique_vs_sweep_$(sym_tag(sym))_$(σ_name).png")
    display(plt)
end
